# RAG Pipeline Execution and Performance Evaluation
This notebook compiles the complete RAG pipeline, runs benchmark evaluations using our custom LLM-as-a-judge evaluators, compares latencies, and plots results.

In [ ]:
import sys
import pandas as pd
sys.path.append('..')
from src.document_processor import DocumentProcessor
from src.retriever import Retriever
from src.rag_engine import RAGEngine
from src.evaluator import RAGEvaluator
print("All pipeline classes loaded successfully!")

### Load the LLM and Evaluator System
We initialize the engine with the lightweight `Qwen/Qwen2.5-0.5B-Instruct` model.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
engine = RAGEngine(model_name=model_name)
evaluator = RAGEvaluator(engine.model, engine.tokenizer)
print("LLM loaded and ready for inference and evaluation.")

### Running a Single RAG Query and Evaluation
Let's run a test query, retrieve chunks using the Hybrid method, generate an answer with reordering, and evaluate its quality.

In [ ]:
processor = DocumentProcessor()
chunks = processor.process_directory("../data", chunk_size=300, chunk_overlap=30)
retriever = Retriever(chunks)

query = "What is the proper patient sitting posture before measuring blood pressure?"
retrieved = retriever.retrieve(query, method="hybrid", top_k=3)

# Generate
answer, context, inf_time = engine.generate_answer(query, retrieved, post_processing="reorder")
print(f"--- Generated Answer (Inference Time: {inf_time:.2f}s) ---\n", answer)

# Evaluate
faith_score, faith_audit = evaluator.evaluate_faithfulness(context, answer)
rel_score, rel_audit = evaluator.evaluate_relevance(query, answer)
print(f"\nEvaluation Metrics:\n- Faithfulness: {faith_score:.2f}\n- Relevance: {rel_score:.2f}")

### Loading benchmark results and plotting comparison charts
We read the experiment results CSV and plot Faithfulness vs Relevance and Latencies.

In [ ]:
import matplotlib.pyplot as plt
import os

if os.path.exists("../data/results.csv"):
    df = pd.read_csv("../data/results.csv")
    print(df.to_string())
else:
    print("Run `run_experiments.py` first to output results CSV!")